# Inteligencia Artificial — Práctica de Laboratorio

## Implementación Manual de K-Nearest Neighbors (KNN) - Nivel Principiante

---

### Información General

| Campo | Detalle |
|---|---|
| **Estudiante** | Alex Guaman |
| **Asignatura** | Inteligencia Artificial / Machine Learning |
| **Tema** | Algoritmo K-Nearest Neighbors (KNN) de forma sencilla |
| **Dataset** | Pacientes Ecuatorianos |

---

### 1. Cargar los Datos
Primero, traemos la información del archivo CSV usando la librería `pandas`.

In [1]:
import pandas as pd
import math
from sklearn.preprocessing import StandardScaler

# Leemos el archivo
df = pd.read_csv('../data/pacientes.csv')
df

,sexo,ciudad,colesterol,edad,diabetes
0,1,Cuenca,bajo,18,no
1,2,Quito,alto,52,si
2,2,Guayaquil,medio,34,no
3,1,Loja,alto,61,si
4,2,Ambato,medio,45,no
5,1,Machala,muy alto,67,si


### 2. Identificar las Variables
Antes de empezar, debemos saber qué tipo de datos tenemos:

| Variable | Tipo |
|---|---|
| `sexo` | Categoría nominal (1 o 2) |
| `ciudad` | Categoría nominal (nombre de la ciudad) |
| `colesterol` | Nivel (bajo, medio, alto, muy alto) |
| `edad` | Número |
| `diabetes` | **Objetivo** (lo que queremos predecir) |

### 3. Transformar los datos a números
Como `sexo` y `ciudad` son variables categóricas nominales, las convertimos con sistema binario (columnas 0/1). El `colesterol` sí se transforma de forma ordinal porque sus niveles tienen orden.

In [2]:
sexo_ciudad_binario = pd.get_dummies(
    df[['sexo', 'ciudad']],
    columns=['sexo', 'ciudad'],
    dtype=int
)

df['colesterol_num'] = df['colesterol'].map({
    'bajo': 1, 'medio': 2, 'alto': 3, 'muy alto': 4
}).astype(int)

df['diabetes_num'] = df['diabetes'].map({'no': 0, 'si': 1}).astype(int)

X = pd.concat([sexo_ciudad_binario, df[['colesterol_num', 'edad']]], axis=1)
columnas_prediccion = X.columns.tolist()
Y = df['diabetes_num']

print("Datos transformados:")
X

Datos transformados:


,sexo_1,sexo_2,ciudad_Ambato,ciudad_Cuenca,ciudad_Guayaquil,ciudad_Loja,ciudad_Machala,ciudad_Quito,colesterol_num,edad
0,1,0,0,1,0,0,0,0,1,18
1,0,1,0,0,0,0,0,1,3,52
2,0,1,0,0,1,0,0,0,2,34
3,1,0,0,0,0,1,0,0,3,61
4,0,1,1,0,0,0,0,0,2,45
5,1,0,0,0,0,0,1,0,4,67


### 4. Estandarización (Poner todo en la misma escala)
Como la `edad` llega hasta 67 y el `colesterol` solo hasta 4, debemos equilibrarlos para que el algoritmo no se confunda.

In [3]:
scaler = StandardScaler()

X_escalado = pd.DataFrame(
    scaler.fit_transform(X),
    columns=columnas_prediccion
)

print("Datos escalados (estandarizados):")
X_escalado

Datos escalados (estandarizados):


,sexo_1,sexo_2,ciudad_Ambato,ciudad_Cuenca,ciudad_Guayaquil,ciudad_Loja,ciudad_Machala,ciudad_Quito,colesterol_num,edad
0,1.0,-1.0,-0.447214,2.236068,-0.447214,-0.447214,-0.447214,-0.447214,-1.566699,-1.708466
1,-1.0,1.0,-0.447214,-0.447214,-0.447214,-0.447214,-0.447214,2.236068,0.522233,0.353824
2,-1.0,1.0,-0.447214,-0.447214,2.236068,-0.447214,-0.447214,-0.447214,-0.522233,-0.737976
3,1.0,-1.0,-0.447214,-0.447214,-0.447214,2.236068,-0.447214,-0.447214,0.522233,0.899725
4,-1.0,1.0,2.236068,-0.447214,-0.447214,-0.447214,-0.447214,-0.447214,-0.522233,-0.070765
5,1.0,-1.0,-0.447214,-0.447214,-0.447214,-0.447214,2.236068,-0.447214,1.566699,1.263658


### 5. Algoritmo KNN Manual
Creamos una función para predecir si un nuevo paciente tiene diabetes.

In [4]:
def predecir_diabetes(nuevo_sexo, nueva_ciudad, nuevo_colesterol, nueva_edad):
    colesterol_n = {'bajo': 1, 'medio': 2, 'alto': 3, 'muy alto': 4}[nuevo_colesterol]
    
    nuevo_df = pd.DataFrame([{
        'sexo': nuevo_sexo,
        'ciudad': nueva_ciudad,
        'colesterol_num': colesterol_n,
        'edad': nueva_edad
    }])
    
    nuevo_binario = pd.get_dummies(
        nuevo_df[['sexo', 'ciudad']],
        columns=['sexo', 'ciudad'],
        dtype=int
    )
    nuevo_p = pd.concat([nuevo_binario, nuevo_df[['colesterol_num', 'edad']]], axis=1)
    nuevo_p = nuevo_p.reindex(columns=columnas_prediccion, fill_value=0)
    
    nuevo_p_escalado = pd.Series(
        scaler.transform(nuevo_p)[0],
        index=columnas_prediccion
    )
    
    todas_las_distancias = []
    
    for i in range(len(X_escalado)):
        paciente_entrenamiento = X_escalado.iloc[i]
        
        suma_cuadrados = 0
        for j in range(len(columnas_prediccion)):
            suma_cuadrados += (nuevo_p_escalado.iloc[j] - paciente_entrenamiento.iloc[j])**2
        
        distancia = math.sqrt(suma_cuadrados)
        todas_las_distancias.append((distancia, i, Y.iloc[i]))
    
    todas_las_distancias.sort()
    vecinos = todas_las_distancias[:3]
    
    votos_si = 0
    votos_no = 0
    
    print("--- Distancias Calculadas ---")
    for d, indice, clase in todas_las_distancias:
        print(f"Paciente {indice}: Distancia: {d:.4f} | Diabetes: {'si' if clase == 1 else 'no'}")
    
    print("\n--- Vecinos Seleccionados (k=3) ---")
    for d, indice, clase in vecinos:
        print(f"Paciente {indice}: Distancia: {d:.4f} | Diabetes: {'si' if clase == 1 else 'no'}")
        if clase == 1:
            votos_si += 1
        else:
            votos_no += 1
            
    if votos_si > votos_no:
        return "SÍ tiene diabetes"
    else:
        return "NO tiene diabetes"

resultado = predecir_diabetes(nuevo_sexo=2, nueva_ciudad='Cuenca', nuevo_colesterol='alto', nueva_edad=50)
print("Resultado:", resultado)


--- Distancias Calculadas ---
Paciente 1: Distancia: 3.7967 | Diabetes: si
Paciente 4: Distancia: 3.9475 | Diabetes: no
Paciente 0: Distancia: 4.0163 | Diabetes: no
Paciente 2: Distancia: 4.0537 | Diabetes: no
Paciente 3: Distancia: 4.7797 | Diabetes: si
Paciente 5: Distancia: 4.9552 | Diabetes: si

--- Vecinos Seleccionados (k=3) ---
Paciente 1: Distancia: 3.7967 | Diabetes: si
Paciente 4: Distancia: 3.9475 | Diabetes: no
Paciente 0: Distancia: 4.0163 | Diabetes: no
Resultado: NO tiene diabetes


---

### 6. Conclusiones

1.  **Importancia del Preprocesamiento:** Se demostró que transformar variables categóricas nominales como `sexo` y `ciudad` con columnas binarias evita crear un orden falso entre categorías.
2.  **Efecto de la Estandarización:** Sin el escalado (Z-score), variables con rangos grandes como la `edad` dominarían el cálculo de la distancia, ignorando otras variables importantes. Al estandarizar, todas las variables tienen el mismo peso.
3.  **Lógica del Algoritmo KNN:** El modelo predijo que el paciente **NO tiene diabetes** basándose en que 2 de sus 3 vecinos más cercanos tampoco tenían la enfermedad.
4.  **Aprendizaje Manual:** Implementar el algoritmo paso a paso permite entender que detrás de las librerías complejas solo hay matemáticas básicas como la distancia euclídea y votaciones por mayoría.